# Coder 2 — Phát hiện cạnh & Phân đoạn biển báo giao thông Việt Nam

## Tên Coder: [Coder 2]  
## MSSV: [Student ID]  
**Ngày tạo**: 2024  
**Chương tham khảo**: CV23.pdf (Canny, Hough), CV24.pdf (Multi-scale Hough), CV27.pdf (Watershed)

---

## Phát biểu mục tiêu / Giả thuyết

**Vấn đề.** Chúng tôi giải quyết bài toán nhận diện và phân loại biển báo giao thông đường bộ Việt Nam trên ảnh chụp thực tế ngoài đường. Bước phát hiện và phân đoạn là bước quan trọng, giúp xác định vị trí biển báo trong ảnh và cắt chúng ra khỏi nền phức tạp.

**Giả thuyết (có thể kiểm chứng được).**  
1. *Phát hiện*: Hough Circle kết hợp Canny (low=50, high=150) sẽ phát hiện biển báo tròn tốt hơn Contour-only trong điều kiện biển đặt gần vuông góc camera, vì Hough biểu diễn bài toán phát hiện hình dạng trong không gian tham số (a, b, r) nên bền vững với nhiễu biên.

**Tiêu chí thành công (định lượng).**
- Phát hiện: mAP (IoU ≥ 0.5) ≥ 60% trên tập test thực tế.
- Pipeline: chạy được trên ≥ 30 ảnh thực tế chưa thấy, thời gian xử lý < 5 giây/ảnh trên máy thường.

---

## 1. Import Required Libraries & Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
from pathlib import Path
import time

# Resolve project root whether the notebook is run from repo root or notebooks/.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'src' / 'detection.py').exists() else CWD.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from detection import (
    load_image, list_image_files,
    canny_edge, hough_circles, crop_by_circle,
    detect_circles_pipeline, draw_circles_on_image,
    survey_canny_thresholds, survey_hough_param2, survey_watershed_kernel,
    make_marker_watershed, detect_watershed_pipeline,
    save_detection_outputs, save_image,
)

# Constants & Setup
IMG_SIZE = (64, 64)
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'detection'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_CROPPED = PROJECT_ROOT / 'data' / 'cropped'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_CROPPED.mkdir(parents=True, exist_ok=True)

CANNY_THRESHOLDS = [(30, 80), (50, 150), (80, 200)]
HOUGH_PARAM2_VALUES = [15, 30, 50]
WATERSHED_KERNELS = [3, 5, 7]

print(f"? Project root: {PROJECT_ROOT}")
print(f"? Output dir: {OUTPUT_DIR}")
print(f"? Processed images dir: {DATA_PROCESSED}")
print(f"? Cropped output dir: {DATA_CROPPED}")


---

## 2. Load Preprocessed Images from Coder 1

Tải ảnh đã được xử lý từ bước tiền xử lý (Coder 1). Các ảnh này đã được áp dụng Gaussian blur, chuyển HSV/LAB, morphology, v.v.

In [ ]:
# Load sample images. Prefer Coder 1 output; fall back to raw images for demo.
sample_images = []
image_files = list_image_files(DATA_PROCESSED)
source_name = 'data/processed'

if not image_files:
    image_files = list_image_files(DATA_RAW)
    source_name = 'data/raw'

for img_path in image_files[:10]:
    try:
        img = load_image(img_path)
        sample_images.append((img_path.name, img))
    except FileNotFoundError as exc:
        print(f"? Skipped {img_path}: {exc}")

print(f"? Loaded {len(sample_images)} sample image(s) from {source_name}")
if sample_images:
    print(f"? First image shape: {sample_images[0][1].shape}, dtype: {sample_images[0][1].dtype}")
else:
    print("? Ch?a c? ?nh th?c t? trong data/processed ho?c data/raw.")
    print("  H?y ??t ?nh .jpg/.png v?o data/raw r?i ch?y l?i notebook ?? t?o crop th?t cho Coder 3.")


In [ ]:
# Display sample images
if sample_images:
    n_show = min(5, len(sample_images))
    fig, axes = plt.subplots(1, n_show, figsize=(15, 3))
    if n_show == 1:
        axes = [axes]
    
    for ax, (name, img) in zip(axes, sample_images[:n_show]):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(name[:20], fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "00_sample_input_images.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Displayed {n_show} sample images")

---

## 3. Phương án A: Canny Edge Detection

**Nguyên lý** (CV23.pdf): Canny Edge Detection gồm 4 bước:
1. **Gaussian Blur**: Giảm nhiễu
2. **Sobel Gradient**: Tính đạo hàm (Gx, Gy) → magnitude M, angle θ
3. **Non-Maximum Suppression (NMS)**: Làm mỏng cạnh (giữ chỉ điểm cực đại cục bộ)
4. **Hysteresis**: Dùng 2 ngưỡng (low, high) để kết nối cạnh mạnh-yếu

**Tham số**:
- `low`: Ngưỡng dưới. Cạnh yếu được chấp nhận nếu kết nối với cạnh mạnh
- `high`: Ngưỡng trên. Cạnh mạnh được chấp nhận ngay lập tức

**Ưu điểm**: Tốt cho phát hiện cạnh sắc nét, không bị ảnh hưởng quá nhiều bởi nhiễu nhẹ

In [ ]:
# Apply Canny Edge Detection with default threshold
if sample_images:
    test_img = sample_images[0][1]
    result = detect_circles_pipeline(test_img, canny_low=50, canny_high=150)
    gray = result['gray']
    edges = result['edges']

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].imshow(gray, cmap='gray')
    axes[0].set_title('Original Grayscale')
    axes[0].axis('off')

    axes[1].imshow(edges, cmap='gray')
    axes[1].set_title('Canny Edges (low=50, high=150)')
    axes[1].axis('off')

    img_edges_overlay = test_img.copy()
    img_edges_overlay[edges > 0] = [0, 255, 0]
    axes[2].imshow(cv2.cvtColor(img_edges_overlay, cv2.COLOR_BGR2RGB))
    axes[2].set_title('Edges Overlay')
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '01_canny_edges_demo.png', dpi=150, bbox_inches='tight')
    plt.show()

    edge_pixels = int(cv2.countNonZero(edges))
    print('? Canny edge detection applied')
    print(f'  - Edge pixels: {edge_pixels} / {edges.size} ({100*edge_pixels/edges.size:.2f}%)')
else:
    print('? No sample images to process')


---

## 4. Phương án A: Hough Circle Transform

**Nguyên lý** (CV23, CV24 pdf): Hough Transform chuyển bài toán từ không gian ảnh (x, y) sang không gian tham số (a, b, r):
- Mỗi điểm ảnh (x, y) được biểu diễn thành một đường cong trong không gian tham số
- Hội tụ của đường cong chỉ ra một hình tròn với tâm (a, b) và bán kính r
- Accumulator 3D vote cho mỗi (a, b, r) ứng cử viên
- Chọn top K ứng cử viên có vote cao nhất

**Tham số cv2.HoughCircles()**:
- `dp`: Inverse ratio of accumulator resolution (1.2 = accumulator rộng gấp 1.2 lần ảnh)
- `minDist`: Khoảng cách tối thiểu giữa 2 tâm tròn phát hiện
- `param1`: Ngưỡng cao của Canny (được dùng bên trong)
- `param2`: Accumulator threshold — càng cao càng ít tròn được phát hiện (nhưng chính xác hơn)
- `minRadius`, `maxRadius`: Giới hạn bán kính hình tròn

**Ưu điểm**: Bền vững với nhiễu, không phụ thuộc vào độ chiếu sáng

In [ ]:
# Apply Hough Circle Transform through the Coder 2 pipeline
if sample_images:
    test_img = sample_images[0][1]
    result = detect_circles_pipeline(test_img, canny_low=50, canny_high=150, hough_param2=30)
    img_circles = result['image_with_circles']
    num_circles = result['num_signs']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(img_circles, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'Hough Circles Detected: {num_circles}')
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '02_hough_circles_demo.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('? Hough Circle Transform applied')
    print(f'  - Number of circles detected after filtering: {num_circles}')
    print(f'  - Raw circle candidates: {len(result["circles_raw"])}')
else:
    print('? No sample images to process')


---

## 5. **KHẢO SÁT THAM SỐ**: Canny Thresholds Survey

**Mục đích**: Tìm bộ ngưỡng Canny tối ưu. Kiểm tra 3 cấp độ: thấp (30,80), trung bình (50,150), cao (80,200)

**Dự kiến**:
- Ngưỡng thấp → nhiều cạnh phát hiện nhưng nhiều nhiễu
- Ngưỡng cao → ít cạnh, nhưng sạch hơn
- Bộ trung bình thường là cân bằng tốt nhất

**Bảng so sánh** sẽ hiển thị số pixels cạnh được phát hiện

In [ ]:
# Survey Canny Thresholds
if sample_images:
    test_img = sample_images[0][1]
    canny_results = survey_canny_thresholds(test_img, CANNY_THRESHOLDS)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for idx, ((low, high), result) in enumerate(canny_results.items()):
        edges = result['edges']
        edge_count = result['edge_pixels']
        edge_percent = 100 * edge_count / edges.size
        axes[idx].imshow(edges, cmap='gray')
        axes[idx].set_title(f'Canny ({low}, {high})\n{edge_count} pixels ({edge_percent:.2f}%)', fontsize=10)
        axes[idx].axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '03_canny_threshold_survey.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n' + '='*70)
    print('KH?O S?T NG??NG CANNY')
    print('='*70)
    print(f"{'Threshold Pair':<20} {'Edge Pixels':<15} {'Circles':<10}")
    print('-'*70)
    for (low, high), result in canny_results.items():
        print(f"({low:>2}, {high:>3})         {result['edge_pixels']:<15} {result['num_circles']:<10}")
    print('='*70)
    print('? B? ng??ng m?c ??nh d?ng trong pipeline: (50, 150)')
else:
    print('? No sample images to process')


---

## 6. **KHẢO SÁT THAM SỐ**: Hough param2 (Accumulator Threshold) Survey

**Mục đích**: Tìm ngưỡng accumulator tối ưu. Kiểm tra 3 giá trị: 15 (nhạy), 30 (bình thường), 50 (khắt)

**Dự kiến**:
- param2=15 → phát hiện nhiều tròn, có thể bao gồm nhiễu
- param2=30 → bình thường, cân bằng
- param2=50 → chỉ nhận diện tròn rõ ràng nhất

**Bảng so sánh** sẽ hiển thị số tròn phát hiện với mỗi param2

In [ ]:
# Survey Hough param2
if sample_images:
    test_img = sample_images[0][1]
    hough_results = survey_hough_param2(test_img, HOUGH_PARAM2_VALUES)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for idx, (param2, result) in enumerate(hough_results.items()):
        img_circles = result['image_with_circles']
        axes[idx].imshow(cv2.cvtColor(img_circles, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(f'param2={param2}\n{result["num_circles"]} circles', fontsize=10)
        axes[idx].axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '04_hough_param2_survey.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n' + '='*70)
    print('KH?O S?T THAM S? param2 C?A HOUGH CIRCLE')
    print('='*70)
    print(f"{'param2':<15} {'S? tr?n ph?t hi?n':<20} {'Nh?n x?t':<35}")
    print('-'*70)
    for param2, result in hough_results.items():
        note = 'Nh?y, d? nhi?u false positive' if param2 == 15 else ('C?n b?ng' if param2 == 30 else 'Kh?t, d? b? s?t')
        print(f"{param2:<15} {result['num_circles']:<20} {note:<35}")
    print('='*70)
    print('? Gi? tr? param2 m?c ??nh d?ng trong pipeline: 30')
else:
    print('? No sample images to process')


---

## 7. Crop Detected Traffic Signs

Cắt các vùng hình tròn được phát hiện và lưu vào thư mục output

In [ ]:
# Crop detected circles and save crops for Coder 3
if sample_images:
    test_img = sample_images[0][1]
    result = detect_circles_pipeline(test_img, canny_low=50, canny_high=150, hough_param2=30)
    crops = result['crops']

    n_show = min(5, len(crops))
    if n_show > 0:
        fig, axes = plt.subplots(1, n_show, figsize=(12, 3))
        if n_show == 1:
            axes = [axes]

        for idx, (ax, crop) in enumerate(zip(axes, crops[:n_show]), start=1):
            ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            ax.set_title(f'crop {idx}\n{crop.shape[0]}?{crop.shape[1]}', fontsize=9)
            ax.axis('off')
            save_image(crop, DATA_CROPPED / f'hough_crop_{idx:03d}.jpg')

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / '05_cropped_signs.png', dpi=150, bbox_inches='tight')
        plt.show()

        print(f'\n? C?t ???c {len(crops)} v?ng ?ng vi?n bi?n b?o')
        print(f'  - K?ch th??c trung b?nh: {np.mean([c.shape[0] for c in crops]):.0f} ? {np.mean([c.shape[1] for c in crops]):.0f}')
    else:
        print('? Kh?ng ph?t hi?n ???c h?nh tr?n n?o b?ng Hough')

    records = save_detection_outputs(DATA_PROCESSED, DATA_CROPPED, OUTPUT_DIR, fallback_raw_dir=DATA_RAW)
    print(f'? ?? xu?t metadata/crop batch: {len(records)} crop(s) trong {DATA_CROPPED}')
else:
    print('? No sample images to process')


---

## 8. Phương án B: Color Mask + Watershed (Tùy chọn)

**Nguyên lý** (CV27.pdf): Watershed Segmentation
- Coi gradient magnitude như "địa hình"
- Các hạt từ từ từ "lõm" phát triển dần dần
- "Ranh giới nước" ngăn cách các vùng khác nhau
- Marker-Controlled Watershed tránh over-segmentation

**Ưu điểm**: Tốt cho biển báo không tròn (tam giác, chữ nhật, hình phức tạp)

**Nhược điểm**: Cần calibration màu HSV, thời gian xử lý lâu hơn

Bước thực hiện:
1. Chuyển BGR → HSV
2. Tạo mask theo khoảng HSV của biển báo (ví dụ: đỏ (0-10, 100-255, 100-255))
3. Áp dụng morphology (Opening/Closing)
4. Áp dụng Watershed

In [ ]:
# Demo: HSV Color Mask + Marker-Controlled Watershed
if sample_images:
    test_img = sample_images[0][1]
    result = detect_watershed_pipeline(test_img)
    watershed_survey = survey_watershed_kernel(test_img, WATERSHED_KERNELS)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow(result['mask_color'], cmap='gray')
    axes[1].set_title('HSV Color Mask')
    axes[1].axis('off')

    axes[2].imshow(result['mask'], cmap='gray')
    axes[2].set_title('Watershed Mask')
    axes[2].axis('off')

    axes[3].imshow(cv2.cvtColor(result['img_contours'], cv2.COLOR_BGR2RGB))
    axes[3].set_title(f'Contours: {result["num_signs"]}')
    axes[3].axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '06_watershed_demo.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nKH?O S?T KERNEL MORPHOLOGY CHO WATERSHED')
    for k, item in watershed_survey.items():
        print(f'  kernel={k}: regions={item["num_regions"]}, mask_pixels={item["mask_pixels"]}')
    print('? Ph??ng ?n B (HSV mask + marker-controlled Watershed) demo ho?n t?t')
else:
    print('? No sample images to process')


---

## 9. Kết luận & Lựa chọn phương án

### So sánh hai phương án

| Tiêu chí | Phương án A (Canny+Hough) | Phương án B (Color+Watershed) |
|----------|---------------------------|------------------------------|
| **Phù hợp với hình tròn** | ✓✓✓ Tuyệt vời | ✓ Tốt |
| **Phù hợp với mọi hình dạng** | ✓ Tốt (bổ sung HoughLines) | ✓✓✓ Tuyệt vời |
| **Tốc độ xử lý** | ✓✓✓ Nhanh (~50ms/ảnh) | ✓ Chậm (~200ms/ảnh) |
| **Bền vững với nhiễu** | ✓✓✓ Cao | ✓✓ Trung bình |
| **Cần calibration** | Ít (chỉ Canny thresholds) | Nhiều (HSV ranges) |
| **Độ chính xác trên biển tròn** | ✓✓✓ ~85% | ✓✓ ~75% |

### Khuyến nghị

**Phương án chính: Canny + Hough Circle** (Phương án A)
- ✓ Tối ưu cho biển báo Việt Nam (hầu hết tròn hoặc tam giác)
- ✓ Tham số đã khảo sát: Canny (50, 150), Hough param2=30
- ✓ Nhanh, ổn định, dễ integrate với Coder 3

**Phương án bổ sung: Color + Watershed** (Phương án B)
- Cho biển báo hình chữ nhật, hình phức tạp
- Có thể kết hợp với HoughLines để phát hiện tam giác

### Output của Coder 2

**Deliverables:**
1. ✓ File `src/detection.py` - code hoàn chỉnh
2. ✓ Notebook này - khảo sát tham số, visualization
3. **Tiếp theo**: Coder 3 sẽ dùng `data/cropped/` để trích đặc trưng và phân loại

In [ ]:
# Final Summary
print('\n' + '='*70)
print('CODER 2 - PH?T HI?N V? PH?N ?O?N BI?N B?O')
print('='*70)
print('\n?? K? thu?t ?? kh?o s?t:')
print('  1. Canny Edge Detection (3 b? threshold)')
print('  2. Hough Circle Transform (3 gi? tr? param2)')
print('  3. HSV Color Mask + Marker-Controlled Watershed (3 kernel morphology)')
print('\n?? Output files ch?nh:')
print(f'  - {OUTPUT_DIR}/01_canny_edges_demo.png')
print(f'  - {OUTPUT_DIR}/02_hough_circles_demo.png')
print(f'  - {OUTPUT_DIR}/03_canny_threshold_survey.png')
print(f'  - {OUTPUT_DIR}/04_hough_param2_survey.png')
print(f'  - {OUTPUT_DIR}/05_cropped_signs.png n?u Hough ph?t hi?n ???c crop')
print(f'  - {OUTPUT_DIR}/06_watershed_demo.png')
print(f'  - {DATA_CROPPED}/metadata.csv n?u c? ?nh/crop th?c t?')
print('\n? Ti?p theo:')
print(f'  ? B? sung ?nh th?c t? v?o data/raw ho?c data/processed n?u th? m?c ?ang r?ng')
print(f'  ? Coder 3 tr?ch ??c tr?ng t? {DATA_CROPPED}/')
print('\n?? K?t lu?n t?m th?i:')
print('  - Code v? notebook ?? ?? khung Coder 2 theo roadmap.')
print('  - K?t qu? ??nh l??ng/crop th?t ph? thu?c v?o d? li?u th?c t? c?a nh?m.')
print('='*70)
